# 00. Imports

In [ ]:
# 🔧 Standard library
import json
import os
import cv2

# 📦 Core PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 🖼️ Torchvision
import torchvision
import torchvision.transforms as T
from torchvision import transforms
from torchvision.transforms import Compose, ToTensor


# 📊 Data handling
import pandas as pd
from sklearn.model_selection import train_test_split

# 📁 Project-specific modules
from src.helper import get_device, draw_boxes_on_image, save_patches
from src.loader import (
    ChocolatePatchDataset,
    CocoDataset,
    PatchTestDataset,
    UnlabeledImageFolder,
    VOCDataset
)
from models.cnn import SimpleCNN
from models.mobile import LightFasterRCNNMobileNetV3
from src.trainer import Trainer


# ⚙️ Device setup
device = get_device()
print(f"[INFO] Using device: {device}")

# 📁 Dataset paths
DATASET_PATH = "dataset_project_iapr2025"
VOC_DATASET_PATH = f"{DATASET_PATH}_voc"

# **01. SSDLiteMobileNetV3 Model** 

## 01.a Download VOC Dataset

In [ ]:
class_names = [
    "Jelly White", "Jelly Milk", "Jelly Black", "Amandina", "Crème brulée",
    "Triangolo", "Tentation noir", "Comtesse", "Noblesse", "Noir authentique",
    "Passion au lait", "Arabia", "Stracciatella"
]

# Example label mapping (fill this in based on your dataset)
label_map = {
    "Jelly_White": 1,
    "Jelly_Milk": 2,
    "Jelly_Black": 3,
    "Amandina": 4,
    "Creme_brulee": 5,
    "Triangolo": 6,
    "Tentation_noir": 7,
    "Comtesse": 8,
    "Noblesse": 9,
    "Noir_authentique": 10,
    "Passion_au_lait": 11,
    "Arabia": 12,
    "Stracciatella": 13
}

label_map_inv = {v: k for k, v in label_map.items()}

In [ ]:
transform = Compose([
    ToTensor()
])

### Train/Val Dataset
full_dataset = VOCDataset(f"{VOC_DATASET_PATH}/train", transform=transform, label_map=label_map)

# check if path exists
if not os.path.exists(f"{VOC_DATASET_PATH}/train"):
    raise FileNotFoundError("Dataset path does not exist.")

# Optional: Split into train/val
train_len = int(0.8 * len(full_dataset))
val_len = len(full_dataset) - train_len
train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

### Test Dataset
test_dir = f"{DATASET_PATH}/test"
test_dataset = UnlabeledImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print("Train/Val split:", train_len, "/", val_len)
print("Test dataset size:", len(test_loader.dataset))

## 01.b Define the model

In [ ]:
# Define the number of classes (including background)
num_classes = 14  # Example: 1 class (e.g., 'chocolate') + 1 background

# Initialize the model without pretrained weights
model = torchvision.models.detection.ssdlite320_mobilenet_v3_large(weights=None)

# Replace the classifier with a new one for your number of classes
model.head.classification_head.num_classes = num_classes

## 01.c Train the model

In [ ]:
# Define the optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 30

trainer = Trainer(model=model,
                  model_name="SSDLiteMobileNetV3",
                      optimizer=optimizer,
                      scheduler=scheduler,
                      num_epochs =num_epochs,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)
trainer.train()
trainer.save_model()

## 01.d Evaluate the model

In [ ]:
avg_loss, overall_accuracy, overall_f1, per_class_f1, _ = trainer.evaluate()
print(f"Average Loss: {avg_loss}, Overall Accuracy: {overall_accuracy}%, Overall F1: {overall_f1}")

per_class_f1 = per_class_f1[1:]  # Drop the background class

for i, f1 in enumerate(per_class_f1):
    print(f"Class {i+1}: {class_names[i]} - F1 Score: {f1}")

## 02.e Create Patches from test dataset for CNN classification

In [ ]:
# Load the model if it is already trained
model = trainer.load_model(DIR="checkpoints/", model_name="SSDLiteMobileNetV3")

In [ ]:
preds = trainer.predict()

In [ ]:
output_image_dir = "results/drawn_images"
output_patch_dir = f"{DATASET_PATH}/test_patches"
os.makedirs(output_image_dir, exist_ok=True)
os.makedirs(output_patch_dir, exist_ok=True)

for pred in preds:
    filename = pred["filename"]
    boxes = pred["boxes"]
    labels = pred["labels"]
    scores = pred["scores"]

    # Save image with boxes - log to see the output
    #drawn = draw_boxes_on_image(filename, boxes, labels, scores, label_map_inv)
    #cv2.imwrite(filename, drawn)

    # Save patches
    save_patches(filename, boxes, labels, scores, label_map_inv, output_patch_dir, threshold=0.1)

# **02. Train Simple CNN**

## 02.a Download patched dataset

In [ ]:
# === 1. Load CSV and create label map ===
csv_path = f"{DATASET_PATH}/train_patches/patches/labels.csv"
image_dir = f"{DATASET_PATH}/train_patches/patches"

# label preprocessing and dataset splitting.
df = pd.read_csv(csv_path, dtype={'name': str})
class_names = sorted(df['label'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
df['class_idx'] = df['label'].map(class_to_idx)

# === 2. Train/val split ===
# stratified sampling to ensure each class is evenly represented in both sets
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['class_idx'], random_state=42)

# move labels 1 up
train_df['class_idx'] = train_df['class_idx'] + 1
val_df['class_idx'] = val_df['class_idx'] + 1

# === 3. Define transform ===
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# === 5. Create datasets and loaders ===
### Train/Val Dataset
train_dataset = ChocolatePatchDataset(train_df, image_dir, transform)
val_dataset = ChocolatePatchDataset(val_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

### Test Dataset
test_dir = f"{DATASET_PATH}/test_patches"
test_dataset = PatchTestDataset(test_dir, transform=transform)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# === 6. Check dataset size ===
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
# === 7. Check first label ===
print("First label in train dataset:", train_dataset[0][1].item())

## 02.b Define the model

In [ ]:
class_number = 14
model = SimpleCNN(input_shape=3, hidden_units=64, image_height=128, image_width=128, output_shape=class_number)
loss_fn = nn.CrossEntropyLoss()

## 02.c Train the model

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

trainer = Trainer(model=model,
                  model_name="SimpleCNN",
                      loss_fn=loss_fn,
                      optimizer=optimizer,
                      scheduler=scheduler,
                      num_epochs=100,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
trainer.save_model()

## 02.d Evaluate the model

In [ ]:
avg_loss, overall_accuracy, overall_f1, per_class_f1 = trainer.evaluate()
print(f"Average Loss: {avg_loss}, Overall Accuracy: {overall_accuracy}%, Overall F1: {overall_f1}")

for i, f1 in enumerate(per_class_f1):
    print(f"Class {i+1}: {class_names[i]} - F1 Score: {f1}")

# **Sameh Training**

# **03. Train MobileNetV3 Model**

## 03.a Download the dataset

In [ ]:
# Define paths and load annotation info
train_img_dir = f"{DATASET_PATH}/train_annotated"
ann_path = f"{DATASET_PATH}/train_annotated/_annotations.coco.json"

# Load COCO annotations and exclude "objects" class
with open(ann_path, "r") as f:
    coco_json = json.load(f)

# Exclude 'objects' and assign label IDs starting from 1
categories = [cat for cat in coco_json["categories"] if cat["name"] != "objects"]
class_name_to_id = {cat["name"]: i + 1 for i, cat in enumerate(categories)}
num_classes = max(class_name_to_id.values()) + 1  # +1 for background class 0

# Display the class structure
print("Detected classes (excluding 'objects'):", list(class_name_to_id.keys()))
print("Total (with background):", num_classes)

In [ ]:
transform = T.Compose([
        T.ToTensor(),  # Converts PIL image to tensor
])

### Train/Val Dataset
# Load full dataset (with all classes including "objects")
full_dataset = CocoDataset(
    root=train_img_dir,
    annotation=ann_path,
    transform=transform
)

# Split full dataset into 90% train / 10% validation
total_len = len(full_dataset)
train_len = int(0.9 * total_len)
val_len = total_len - train_len

train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len])


# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

### Test Dataset
test_image_dir = f"{DATASET_PATH}/test"
unlabeled_dataset = UnlabeledImageFolder(test_image_dir)
test_loader = DataLoader(unlabeled_dataset, batch_size=1, shuffle=False)

print(f"Loaded full dataset: {total_len} images -> {train_len} train / {val_len} val")

## 03.b Define the model

In [ ]:
model = LightFasterRCNNMobileNetV3(num_classes=num_classes)

## 03.c Train the model

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
epochs = 150

trainer = Trainer(model=model,
                  model_name="MobileNetV3",
                      optimizer=optimizer,
                      num_epochs=epochs,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
trainer.save_model()

## 03.d Evaluate the model

In [ ]:
avg_loss, overall_accuracy, overall_f1, per_class_f1 = trainer.evaluate()
print(f"Average Loss: {avg_loss}, Overall Accuracy: {overall_accuracy}%, Overall F1: {overall_f1}")

for i, f1 in enumerate(per_class_f1):
    print(f"Class {i+1}: {class_names[i]} - F1 Score: {f1}")